# 01. Генерация синтетического датасета рецензий (distillation)

Модель-учитель **DeepSeek-V3.2** (Chutes API) размечает статьи из локального архива
eLibrary, результат — SFT-датасет в формате `instruction/input/output`.

Ключевые решения:

* **Асинхронность + семафор**: 20 параллельных запросов, retry с экспоненциальной задержкой;
* **Структурная аугментация 30 % примеров**: модель должна учиться *отклонять* плохие
  работы, а не только хвалить. 15 % статей обрезаются вместе с выводами
  (`truncated_no_conclusions`), 15 % — с перемешанными абзацами (`shuffled_logic`);
* **Валидация JSON** ответа учителя до записи: битые ответы в датасет не попадают;
* **Дозапись + flush** каждой строки: прогон на 1250 статей нельзя терять при падении.

Требуется `pip install -r requirements-train.txt` и `CHUTES_API_TOKEN` в `.env`.
Исходный архив статей (`dataset/data_3`, ~2500 txt) в публичный репозиторий не входит.

In [ ]:
# --- Repo-relative пути: ноутбуки лежат в notebooks/, данные — в корне репозитория ---
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Рабочий каталог:", REPO_ROOT)

## 1. Генерация датасета (DeepSeek-V3 как teacher)

In [ ]:
import os
import json
import re
import random
import logging
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio

# ==========================================
# 1. НАСТРОЙКИ И ЛОГИРОВАНИЕ
# ==========================================
load_dotenv()
CHUTES_API_KEY = os.getenv("CHUTES_API_TOKEN")

if not CHUTES_API_KEY:
    raise ValueError("Ключ CHUTES_API_TOKEN не найден!")

DATA_DIRS = [Path("dataset/data_3"), Path("dataset/data_3_1")]
OUTPUT_JSONL = "synthetic_reviews_dataset_v2.jsonl"
MODEL_NAME = "deepseek-ai/DeepSeek-V3.2-TEE"
CONCURRENT_REQUESTS = 20  # Количество параллельных запросов (ускоряет работу в 20 раз)

# Настраиваем логирование: пишем и в консоль, и в файл
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("dataset_generation.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

# Инициализируем асинхронного клиента
client = AsyncOpenAI(
    api_key=CHUTES_API_KEY,
    base_url="https://llm.chutes.ai/v1"
)

# Семафор для ограничения одновременных запросов (чтобы сервер не забанил за спам)
semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

# ==========================================
# 2. АУГМЕНТАЦИЯ И ОЧИСТКА ТЕКСТА
# ==========================================
def clean_text(text: str) -> str:
    """Оставляем только удаление лишних пробелов, без регулярки на дефисы."""
    return re.sub(r'\s+', ' ', text).strip()

def augment_article(text: str) -> tuple[str, str]:
    """
    С вероятностью 30% "портит" статью, чтобы научить модель отклонять плохие работы.
    Возвращает кортеж: (измененный_текст, тип_аугментации)
    """
    chance = random.random()
    
    if chance < 0.70:
        return text, "original"  # 70% статей идут как есть (надеемся на "Принять" / "Доработать")
    
    elif chance < 0.85:
        # 15% шанса: Удаляем последние 30% статьи (Выводы и результаты)
        words = text.split()
        cut_index = int(len(words) * 0.7)
        truncated_text = " ".join(words[:cut_index])
        return truncated_text, "truncated_no_conclusions"
        
    else:
        # 15% шанса: Жестко перемешиваем абзацы (Ломаем логику и методологию)
        paragraphs = [p for p in text.split('.') if len(p.strip()) > 20]
        random.shuffle(paragraphs)
        shuffled_text = ". ".join(paragraphs) + "."
        return shuffled_text, "shuffled_logic"

# ==========================================
# 3. АСИНХРОННЫЙ ЗАПРОС К API
# ==========================================
async def get_review_from_teacher(article_text: str, file_name: str, retry_count=3) -> str:
    """Асинхронно запрашивает рецензию у DeepSeek. В случае ошибки делает ретраи."""
    system_prompt = """Ты — ведущий научный рецензент уровня PhD. 
Твоя задача — проанализировать научную статью целиком и выдать структурированную объективную рецензию.

ТВОЙ ОТВЕТ ДОЛЖЕН БЫТЬ СТРОГО В ФОРМАТЕ JSON:
{
    "summary": "Краткое описание сути работы (2-3 предложения)",
    "methodology_score": Оценка методологии (целое число от 1 до 10),
    "novelty_score": Оценка научной новизны (целое число от 1 до 10),
    "strengths": ["строго аргументированная сильная сторона 1", "сильная сторона 2"],
    "weaknesses": ["аргументированная слабая сторона 1", "слабая сторона 2"],
    "final_verdict": "Общий вывод: Принять, Отправить на доработку или Отклонить"
}

Выведи ТОЛЬКО валидный JSON, без markdown-разметки (без ```json)."""

    async with semaphore:  # Ждем своей очереди, если 20 потоков уже заняты
        for attempt in range(retry_count):
            try:
                response = await client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": f"ТЕКСТ СТАТЬИ:\n{article_text}"}
                    ],
                    temperature=0.3,
                    max_tokens=1000,
                    stream=False
                )
                return response.choices[0].message.content.strip()
            
            except Exception as e:
                logging.warning(f"[{file_name}] Ошибка API (Попытка {attempt+1}/{retry_count}): {e}")
                await asyncio.sleep(2 ** attempt)  # Экспоненциальная задержка перед повтором
        
        logging.error(f"[{file_name}] Не удалось получить ответ после {retry_count} попыток.")
        return None

# ==========================================
# 4. ОБРАБОТКА ОДНОГО ФАЙЛА
# ==========================================
async def process_file(file_path: Path):
    """Читает, аугментирует, делает запрос и возвращает JSON-строку для записи."""
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw_text = f.read()
            
        cleaned_text = clean_text(raw_text)
        if len(cleaned_text) < 500:
            return None  # Пропускаем пустышки
            
        # Аугментируем текст
        final_text, aug_type = augment_article(cleaned_text)
        
        # Получаем рецензию
        raw_review = await get_review_from_teacher(final_text, file_path.name)
        
        if not raw_review:
            return None

        # Проверяем валидность JSON
        try:
            review_json = json.loads(raw_review)
            
            dataset_row = {
                "instruction": "Проанализируй предоставленный отрывок научной статьи и составь формальную рецензию в формате JSON.",
                "input": final_text,
                "output": json.dumps(review_json, ensure_ascii=False),
                "metadata": {
                    "source_file": file_path.name,
                    "augmentation_type": aug_type
                }
            }
            return json.dumps(dataset_row, ensure_ascii=False)
            
        except json.JSONDecodeError:
            logging.error(f"[{file_path.name}] Модель выдала невалидный JSON.")
            return None
            
    except Exception as e:
        logging.error(f"[{file_path.name}] Внутренняя ошибка обработки: {e}")
        return None

# ==========================================
# 5. ГЛАВНАЯ ФУНКЦИЯ (ОРКЕСТРАТОР)
# ==========================================
async def build_dataset_async(max_samples=2500):
    logging.info("Начинаем сбор путей к файлам...")
    all_files = []
    for data_dir in DATA_DIRS:
        if data_dir.exists():
            for category_dir in data_dir.iterdir():
                if category_dir.is_dir():
                    all_files.extend(list(category_dir.glob("*.txt")))
    
    random.shuffle(all_files)
    files_to_process = all_files[:max_samples]
    logging.info(f"Найдено файлов для обработки: {len(files_to_process)}")

    # Создаем задачи для всех файлов
    tasks = [process_file(f) for f in files_to_process]
    
    successful_count = 0
    # Открываем файл на дозапись. aiofiles тут не обязателен, так как мы пишем по мере готовности
    with open(OUTPUT_JSONL, 'a', encoding='utf-8') as outfile:
        # tqdm_asyncio.gather запускает все задачи и показывает общий прогресс-бар
        for coro in tqdm_asyncio.as_completed(tasks, desc="Генерация рецензий"):
            result = await coro
            if result:
                outfile.write(result + '\n')
                outfile.flush() # Сразу сбрасываем на диск для надежности
                successful_count += 1

    logging.info(f"Сборка датасета завершена! Успешно обработано: {successful_count}/{len(files_to_process)}")

if __name__ == "__main__":
    # Запускаем асинхронный цикл событий
    # Полный прогон остановлен на 1250 записях (см. docs/results.md)
await build_dataset_async(max_samples=2500)

## 2. EDA: распределение аугментаций и вердиктов

Проверяем, что аугментация реально меняет распределение вердиктов: на «порченых»
текстах доля отказов должна быть выше, иначе датасет бесполезен для обучения критике.

In [ ]:
import json
from collections import Counter

from src.config import SYNTHETIC_DATASET

# Полный датасет (68 МБ, 1250 записей) в git не публикуется;
# формат записей показан в data/samples/dataset_sample.jsonl
DATASET_PATH = str(SYNTHETIC_DATASET)

def run_eda():
    total_records = 0
    verdicts = Counter()
    aug_types = Counter()
    
    # Словари для анализа связи "Аугментация -> Вердикт"
    verdicts_by_aug = {
        "original": Counter(),
        "truncated_no_conclusions": Counter(),
        "shuffled_logic": Counter()
    }
    
    methodology_scores = []
    novelty_scores = []
    invalid_json_in_output = 0

    print("=== Чтение и анализ датасета ===")
    
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                total_records += 1
                
                # Достаем метаданные
                aug_type = record.get("metadata", {}).get("augmentation_type", "unknown")
                aug_types[aug_type] += 1
                
                # Пробуем распарсить саму рецензию (output)
                try:
                    review = json.loads(record["output"])
                    
                    verdict = review.get("final_verdict", "Unknown")
                    m_score = review.get("methodology_score", 0)
                    n_score = review.get("novelty_score", 0)
                    
                    # Нормализуем вердикты (на случай если модель написала с маленькой буквы)
                    verdict = verdict.strip().capitalize()
                    
                    verdicts[verdict] += 1
                    methodology_scores.append(m_score)
                    novelty_scores.append(n_score)
                    
                    if aug_type in verdicts_by_aug:
                        verdicts_by_aug[aug_type][verdict] += 1
                        
                except json.JSONDecodeError:
                    invalid_json_in_output += 1
                    
            except json.JSONDecodeError:
                continue

    print(f"\nВсего успешных записей в датасете: {total_records}")
    if invalid_json_in_output > 0:
        print(f"⚠️ ВНИМАНИЕ: Найдено {invalid_json_in_output} записей с битым JSON внутри output.")

    print("\n--- 1. Распределение аугментаций (Ожидаем ~70% / 15% / 15%) ---")
    for aug, count in aug_types.items():
        print(f" - {aug}: {count} ({(count/total_records)*100:.1f}%)")

    print("\n--- 2. Общее распределение вердиктов ---")
    for v, count in verdicts.most_common():
        print(f" - {v}: {count} ({(count/total_records)*100:.1f}%)")

    if methodology_scores and novelty_scores:
        avg_m = sum(methodology_scores) / len(methodology_scores)
        avg_n = sum(novelty_scores) / len(novelty_scores)
        print("\n--- 3. Средние оценки ---")
        print(f" - Методология: {avg_m:.2f} / 10")
        print(f" - Новизна: {avg_n:.2f} / 10")

    print("\n--- 4. Эффективность аугментации (Как порча текста влияет на вердикт) ---")
    for aug, v_counts in verdicts_by_aug.items():
        if sum(v_counts.values()) == 0:
            continue
        print(f"\n[{aug.upper()}]:")
        total_for_aug = sum(v_counts.values())
        for v, count in v_counts.most_common():
            print(f"   - {v}: {count} ({(count/total_for_aug)*100:.1f}%)")

if __name__ == "__main__":
    # Если путь другой, поменяй переменную DATASET_PATH
    DATA_PATH = DATASET_PATH
    run_eda()

## 3. Графики для отчёта

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Настройка стиля для академических графиков
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

def plot_augmentation_pie():
    # Данные из EDA
    labels = ['Оригинальные тексты\n(Без изменений)', 'Нарушение логики\n(Перемешивание абзацев)', 'Удаление выводов\n(Truncated)']
    sizes = [69.0, 15.9, 15.0]
    colors = ['#4C72B0', '#C44E52', '#55A868']
    explode = (0.05, 0, 0)  # Выделяем самую большую часть

    fig, ax = plt.subplots(figsize=(8, 6))
    wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors,
                                      autopct='%1.1f%%', shadow=False, startangle=140,
                                      textprops=dict(color="black", fontsize=12))
    
    plt.setp(autotexts, size=12, weight="bold", color="white")
    # plt.title('Рисунок X. Структура обучающей выборки по типам аугментации', fontsize=14, pad=20, weight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'augmentation_pie.png', dpi=300, bbox_inches='tight')
    print("Сохранен график: augmentation_pie.png")

def plot_verdicts_impact():
    # Данные из EDA
    categories = ['Оригинальные\nстатьи', 'Удаление\nвыводов', 'Нарушение\nлогики']
    
    # Проценты по каждому вердикту внутри категории
    # Порядок: [Original, Truncated, Shuffled]
    accept = [6.6, 4.8, 0.5]
    revise = [88.8, 88.3, 69.3]
    reject = [4.6, 6.9, 30.2]

    x = np.arange(len(categories))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Строим столбцы
    rects1 = ax.bar(x - width, accept, width, label='Принять', color='#55A868')
    rects2 = ax.bar(x, revise, width, label='На доработку', color='#EAE509')
    rects3 = ax.bar(x + width, reject, width, label='Отклонить', color='#C44E52')

    # Настройки осей и надписей
    ax.set_ylabel('Доля вердиктов (%)', fontsize=12, weight="bold")
    # ax.set_title('Рисунок Y. Влияние структурной аугментации текста на вердикт модели-учителя', fontsize=14, pad=20, weight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=12)
    ax.legend(fontsize=11)

    # Добавляем значения над столбцами
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # Сдвиг текста вверх
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'verdicts_bar.png', dpi=300, bbox_inches='tight')
    print("Сохранен график: verdicts_bar.png")

if __name__ == "__main__":
    plot_augmentation_pie()
    plot_verdicts_impact()